# Loss Functions, Cost Functions, and Optimization Objectives

**Session:** 2026-10-31 · Competition tasks and metrics  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

In machine learning literature and competitive data science, the terms **loss function**, **cost function**, **objective function**, and **evaluation metric** are often conflated. However, each term occupies a distinct mathematical role in the training and evaluation lifecycle.

## Theoretical Hierarchy

1. **Loss Function $\mathcal{L}(y_i, \hat{y}_i)$:** A scalar measure of error computed for a **single sample point**.
2. **Cost Function $J(\theta)$:** An aggregate measure (such as sum or mean) computed over a **batch or complete dataset**:
   $$J(\theta) = \frac{1}{N} \sum_{i=1}^N \mathcal{L}(y_i, f(x_i; \theta))$$
3. **Objective Function $\mathcal{J}(\theta)$:** The comprehensive function minimized or maximized during optimization, often augmenting the cost function with regularization terms $\Omega(\theta)$ or constraint penalties:
   $$\mathcal{J}(\theta) = J(\theta) + \lambda \Omega(\theta)$$
4. **Evaluation Metric $M(y, \hat{y})$:** A human-interpretable performance measure used to assess the model's business or competition utility (e.g., Accuracy, F1-Score, IoU). Metrics are often non-differentiable step functions that cannot be directly optimized via gradient descent.

## Learning Outcomes

- Implement the 0-1 misclassification loss and empirical risk cost function.
- Understand why non-differentiable losses cannot be optimized by gradient descent.
- Implement Mean Squared Error (MSE) loss and cost functions.
- Formulate numerically stable Binary Cross-Entropy (BCE) loss with clipping.
- Construct regularized objective functions incorporating L1 (Lasso) and L2 (Ridge) penalties.


In [ ]:
from collections.abc import Sequence
from typing import Any

import numpy as np


# 1. Base 0-1 Loss Function (Pointwise)
def zero_one_loss(y_true: Any, y_pred: Any) -> int:
    """Return 0 if the prediction is correct, otherwise 1."""
    return 0 if y_true == y_pred else 1


# 2. Base Cost Function (Aggregate across dataset)
def empirical_risk_cost(y_true_list: Sequence[Any], y_pred_list: Sequence[Any]) -> float:
    """Compute total misclassification error across dataset."""
    assert len(y_true_list) == len(y_pred_list), "Input sequences must have equal length"
    total_loss = sum(zero_one_loss(yt, yp) for yt, yp in zip(y_true_list, y_pred_list, strict=True))
    return total_loss / len(y_true_list)


actual = ["cat", "dog", "cat", "cat", "dog"]
predicted = ["cat", "cat", "cat", "cat", "dog"]

cost = empirical_risk_cost(actual, predicted)
print(f"Empirical 0-1 Risk (Misclassification rate): {cost:.2%}")
print(f"Accuracy equivalent: {(1.0 - cost):.2%}")

## 1. Why 0-1 Loss Cannot Be Optimized with Gradient Descent

The derivative of the 0-1 loss with respect to continuous model parameters is zero almost everywhere, with undefined step jumps at the decision boundary. Because $\nabla_\theta \mathcal{L}_{0-1} = 0$, gradient descent cannot update model weights. Machine learning practitioners instead optimize smooth, continuous **surrogate losses** such as Cross-Entropy or Mean Squared Error.


In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-np.clip(z, -250, 250)))


# Synthetic margin values (z = y * w^T x)
margins = np.linspace(-3.0, 3.0, 100)
zero_one_curve = (margins < 0).astype(float)
logistic_loss_curve = np.log2(1.0 + np.exp(-margins))

print("At margin = 1.0:")
print(f"  0-1 Loss:        {0.0:.2f} (gradient: 0.0)")
print(
    f"  Log Loss value:  {np.log2(1.0 + np.exp(-1.0)):.4f} (smooth non-zero gradient provides learning signal)"
)

## 2. Regression: Squared Loss vs Mean Squared Error Cost

For continuous outputs, the squared error for sample $i$ is:
$$\mathcal{L}_{\text{SE}}(y_i, \hat{y}_i) = (y_i - \hat{y}_i)^2$$
Aggregating over a batch of size $N$ yields the Mean Squared Error (MSE) cost:
$$J_{\text{MSE}}(\theta) = \frac{1}{N} \sum_{i=1}^N (y_i - f(x_i; \theta))^2$$


In [ ]:
import numpy as np


def pointwise_squared_loss(y_true: float, y_pred: float) -> float:
    return (y_true - y_pred) ** 2


def mse_cost_function(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    assert y_true.shape == y_pred.shape, "Shape mismatch"
    differences = y_true - y_pred
    return float(np.mean(differences**2))


y_actual = np.array([10.5, 14.2, 8.9, 12.0])
y_estimate = np.array([10.2, 15.0, 8.5, 12.3])

print(f"Sample 1 Squared Loss: {pointwise_squared_loss(y_actual[0], y_estimate[0]):.4f}")
print(f"Batch MSE Cost:        {mse_cost_function(y_actual, y_estimate):.4f}")

## 3. Classification: Binary Cross-Entropy with Numerical Stability

The binary cross-entropy loss for true label $y_i \in \{0, 1\}$ and predicted probability $p_i = \hat{y}_i \in (0, 1)$ is:
$$\mathcal{L}_{\text{BCE}}(y_i, p_i) = -\left[ y_i \log(p_i) + (1 - y_i) \log(1 - p_i) \right]$$
When $p_i \to 0$, $\log(p_i) \to -\infty$, triggering `NaN` floating point overflows. We introduce numerical clipping with $\epsilon = 10^{-15}$.


In [ ]:
import numpy as np


def stable_bce_loss(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    eps: float = 1e-15,
) -> float:
    """Numerically stable binary cross-entropy cost function."""
    p_clipped = np.clip(y_prob, eps, 1.0 - eps)
    sample_losses = -(y_true * np.log(p_clipped) + (1.0 - y_true) * np.log(1.0 - p_clipped))
    return float(np.mean(sample_losses))


y_true_binary = np.array([1, 1, 0, 0, 1])
y_prob_good = np.array([0.95, 0.88, 0.05, 0.12, 0.99])
y_prob_poor = np.array([0.20, 0.40, 0.80, 0.70, 0.10])

print(f"BCE Cost (Well-calibrated model): {stable_bce_loss(y_true_binary, y_prob_good):.4f}")
print(f"BCE Cost (Poor model):            {stable_bce_loss(y_true_binary, y_prob_poor):.4f}")

## 4. Constructing Regularized Objective Functions

To prevent overfitting, objective functions penalize model parameter complexity.
- **L1 Regularization (Lasso):** $\Omega(\theta) = \sum_j |w_j|$ (promotes parameter sparsity).
- **L2 Regularization (Ridge):** $\Omega(\theta) = \frac{1}{2} \sum_j w_j^2$ (shrinks weights continuously).

$$\mathcal{J}(\theta) = J_{\text{MSE}}(\theta) + \lambda_1 \sum_{j=1}^D |w_j| + \frac{\lambda_2}{2} \sum_{j=1}^D w_j^2$$


In [ ]:
import numpy as np


def regularized_objective(
    weights: np.ndarray,
    features: np.ndarray,
    targets: np.ndarray,
    l1_lambda: float = 0.0,
    l2_lambda: float = 0.0,
) -> float:
    """Complete objective function combining MSE cost with ElasticNet regularization."""
    predictions = features @ weights
    mse = float(np.mean((targets - predictions) ** 2))

    l1_penalty = l1_lambda * float(np.sum(np.abs(weights)))
    l2_penalty = 0.5 * l2_lambda * float(np.sum(weights**2))

    total_objective = mse + l1_penalty + l2_penalty
    return total_objective


# Demonstrate on dummy weight vectors
dummy_features = np.eye(4)
dummy_targets = np.array([1.0, 2.0, 3.0, 4.0])
w_compact = np.array([1.0, 2.0, 3.0, 4.0])
w_inflated = np.array([10.0, 20.0, -15.0, 8.0])

obj_compact = regularized_objective(w_compact, dummy_features, dummy_targets, l2_lambda=0.1)
obj_inflated = regularized_objective(w_inflated, dummy_features, dummy_targets, l2_lambda=0.1)

print(f"Objective Value (compact weights):  {obj_compact:.4f}")
print(f"Objective Value (inflated weights): {obj_inflated:.4f}")